# 01 - Global dedupe, protected splits, CI/CII experiences (Stages 1 and 1b)
**Order matters (this is what prevents leakage):** dedupe globally on model features -> cap distinct rows per attack class -> split into train/calib/val/test -> *then* pick which train rows arrive in which experience -> fit the scaler on E0 train only.

**Why dedupe first:** the full audit found ~59.9M within-file duplicate rows out of 70.4M. Capping before dedupe would sample mostly copies.

**Experience assignment is separate from embedding.** A manifest is lists of row ids; embeddings are made inside the model per batch.

There are two **STOP** points: after Stage 1 (report counts) and after Stage 1b (report the invariants).

In [ ]:
import glob, json, os, subprocess, sys
from pathlib import Path

# Folder holding s0_audit.py etc. On Kaggle: the uploaded code Dataset.
CODE_DIR = os.environ.get("NIDS_CODE") or next(
    (str(Path(p).parent) for p in glob.glob("/kaggle/input/**/s0_audit.py", recursive=True)), ".")
INPUT = os.environ.get("NIDS_INPUT", "/kaggle/input")   # where the CICDDoS2019 CSVs live
sys.path.insert(0, CODE_DIR)
from common import default_work_dir, find_files
OUT = default_work_dir()
print("code:", CODE_DIR, "| data:", INPUT, "| outputs:", OUT)

def run(script, *args):
    """Run one of our scripts and stream its output (same as `!python script args`)."""
    cmd = [sys.executable, "-u", str(Path(CODE_DIR) / script), *map(str, args)]
    print("$", " ".join(cmd))
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:  # stream: training prints progress as it goes
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"{script} failed (exit {p.returncode})")

## 1. Use the FULL audit manifest
If `/kaggle/working/nids_out/manifest.json` is missing (new session), it is copied from an attached input (e.g. your saved 00 output). A truncated (QUICK) manifest is refused.

In [ ]:
import shutil
import pandas as pd
OUT.mkdir(parents=True, exist_ok=True)
if not (OUT / "manifest.json").exists():
    cands = [Path(p) for p in glob.glob("/kaggle/input/**/manifest.json", recursive=True)]
    cands = [c for c in cands if not any(f["scan_truncated"] for f in json.loads(c.read_text())["files"])]
    assert cands, "no full (untruncated) manifest.json found - run 00_audit with QUICK=False or attach its output"
    shutil.copy(cands[0], OUT / "manifest.json")
    print("copied", cands[0])
man = json.loads((OUT / "manifest.json").read_text())
assert not any(f["scan_truncated"] for f in man["files"]), "truncated manifest - refuse"
on_disk = sorted(map(str, find_files(INPUT)))
assert on_disk == sorted(f["path"] for f in man["files"]), "attached files differ from the audited files"
tot = man["label_totals"]
print(f"{len(man['files'])} files, {sum(tot.values()):,} rows, untruncated")
print(pd.Series(tot).sort_values(ascending=False).to_string())

## 2. Stage 1: one streaming pass (global dedupe -> cap -> split)
- `CAP` = max **distinct** rows kept per attack class (bottom-k on the feature hash = uniform, deterministic). BENIGN: every distinct row kept.
- `NONFINITE="drop"` loses only the ~1.1k benign rows the audit flagged as inf/NaN.
- Runtime: reads every file once, like the full audit, so expect roughly the audit's duration. Progress is printed per file.

In [ ]:
CAP = 60000
SPLIT = "random"
NONFINITE = "drop"
run("s1_preprocess.py", "--input", INPUT, "--cap", CAP, "--split", SPLIT, "--nonfinite", NONFINITE)

## 3. Counts after cleaning, and the overlap refusal
**STOP 1:** paste this cell's output back before choosing the schedule.

In [ ]:
sm = json.loads((OUT / "split_manifest.json").read_text())
st = sm["stats"]
tab = pd.DataFrame({"raw": st["raw_rows_by_label"], "nonfinite": st["nonfinite_rows_by_label"],
                    "distinct": st["distinct_rows_by_label"], "conflict": st["conflict_distinct_by_label"],
                    "kept": st["kept_by_label"]}).fillna(0).astype(int).sort_values("raw", ascending=False)
print(tab.to_string())
print(f"\nDISTINCT BENIGN KEPT: {st['benign_distinct_kept']:,}  (raw was {st['raw_rows_by_label'].get('BENIGN', 0):,})")
cnt = pd.DataFrame(sm["counts_per_split"]).fillna(0).astype(int)
print("\nper-split counts:\n", cnt.to_string())
hs = pd.concat([pd.read_parquet(OUT / f"clean_{sp}.parquet", columns=["_h"])["_h"] for sp in ("train", "calib", "val", "test")])
assert hs.is_unique and sm["cross_split_feature_overlap"] == 0, "REFUSE: identical feature vector in two splits"
print("\nno model-feature vector appears in two splits: OK")
bn = cnt.loc["BENIGN"]
print(f"benign val = {bn['val']:,} -> smallest measurable false-alarm step ~ {1/max(1,bn['val']):.5f}")
for k in (2, 3):
    print(f"CII with {k} experiences -> {bn['train']//k:,} distinct benign train rows each")

## 4. Schedule (decide AFTER reading STOP 1)
Proposed: **E0 = BENIGN + Syn, E1 = LDAP, E2 = UDP**. UDP is then never in E0/E1 training or replay (replay only holds accepted earlier experiences).
`ATTACK_CAP` = train rows per attack class inside its experience. Default rule: about the size of one CII benign portion, at most 20,000, and never more than the class has.
Set `COUNTS_REVIEWED = True` only after the counts above have been checked.

In [ ]:
SCHEDULE = [["Syn"], ["LDAP"], ["UDP"]]
COUNTS_REVIEWED = False
if os.environ.get("NIDS_SCHEDULE"):          # used only by the smoke test
    SCHEDULE, COUNTS_REVIEWED = json.loads(os.environ["NIDS_SCHEDULE"]), True
assert COUNTS_REVIEWED, "STOP 1: report the counts, then set COUNTS_REVIEWED = True"
tr_cnt = cnt["train"]
for c in [k for e in SCHEDULE for k in e]:
    assert c in tr_cnt.index, f"{c} has no rows after cleaning"
benign_portion = int(tr_cnt["BENIGN"]) // len(SCHEDULE)
ATTACK_CAP = int(min(20000, max(benign_portion, 1000), *[tr_cnt[k] for e in SCHEDULE for k in e]))
print("schedule:", SCHEDULE, "| CII benign per experience:", benign_portion, "| ATTACK_CAP:", ATTACK_CAP)
(OUT / "schedule.json").write_text(json.dumps({"experiences": SCHEDULE}))

## 5. Stage 1b: CI and CII manifests + E0-frozen scaler + protected exclusions
Stage 1 made every **raw** feature vector unique. The model sees the **transformed** input, and the transform can map two distinct raw rows to one input: `-0.0` vs `0.0`, values beyond the ±5 clip, differences below float32 precision, or protocol values never seen in E0.

Each run builds **both** scenarios internally and excludes from scoring every calib/val/test row whose model input equals a training input under the CI **or** CII scaler. CI and CII therefore score exactly the same row ids. Only features are compared (no labels, no model), and training rows and the scaler are unchanged.

`protected_exclusions.json` records the excluded rows per split and class, the first transform step that merged each one, which features differed, and a recheck that the remaining overlap is exactly 0.

**Refuses if:**
- any split/class would lose more than 5% (then fix the transform rather than drop rows);
- fewer than 5,000 benign val or test rows remain;
- a scheduled class or a CII benign portion has fewer than 1,000 train rows.

In [ ]:
MIN_ROWS = int(os.environ.get("NIDS_MIN_ROWS", 1000))           # env overrides: smoke test only
MIN_BENIGN = int(os.environ.get("NIDS_MIN_BENIGN", 5000))
for sc in ("ci", "cii"):
    run("s1b_experiences.py", "--schedule", OUT / "schedule.json", "--scenario", sc,
        "--attack-cap", ATTACK_CAP, "--min-rows", MIN_ROWS, "--min-benign-protected", MIN_BENIGN)

## 6. Verify the invariants
**STOP 2:** paste this output back; then go to `02_train_evaluate.ipynb`.

In [ ]:
import numpy as np
tr = pd.read_parquet(OUT / "clean_train.parquet", columns=["_rid", "label"])
protected = set()
for sp in ("calib", "val", "test"):
    protected |= set(pd.read_parquet(OUT / f"clean_{sp}.parquet", columns=["_rid"])["_rid"])
lab = tr.set_index("_rid")["label"]
for sc in ("ci", "cii"):
    z = np.load(OUT / sc / "experience_rids.npz")
    ids = [z[f"e{t}"] for t in range(len(z.files))]
    allids = np.concatenate(ids)
    assert len(allids) == len(set(allids)), f"{sc}: a row is in two experiences"
    assert not (set(allids) & protected), f"{sc}: protected row inside an experience"
    ben = [set(i[lab.loc[i].to_numpy() == "BENIGN"]) for i in ids]
    for a_ in range(len(ben)):
        for b_ in range(a_ + 1, len(ben)):
            assert not (ben[a_] & ben[b_]), f"{sc}: benign shared by E{a_},E{b_}"
    for t, e in enumerate(SCHEDULE):  # later classes never appear in earlier experiences
        for k in e:
            assert not any((lab.loc[ids[u]] == k).any() for u in range(t)), f"{sc}: {k} before E{t}"
    em = json.loads((OUT / sc / "experience_manifest.json").read_text())
    assert all(v["equal_to_a_train_input"] == 0 for v in em["scaled_input_overlap"].values()), f"{sc}: overlap left"
    print(f"{sc}: OK | benign per experience {[len(b) for b in ben]}")
    print("   counts:", em["counts_per_experience"])
    print("   transformed duplicates before exclusion:", em["scaled_input_overlap_before_exclusion"])
    print("   overlap after exclusion:", {k: v["equal_to_a_train_input"] for k, v in em["scaled_input_overlap"].items()})

pe = json.loads((OUT / "protected_exclusions.json").read_text())
shas = {sc: json.loads((OUT / sc / "experience_manifest.json").read_text())["protected_exclusions"]["rids_sha256"]
        for sc in ("ci", "cii")}
assert len(set(shas.values())) == 1 and pe["rids_sha256"] == shas["ci"], "CI and CII exclude different rows"
assert not (set(pe["excluded_rids"]) & set(allids)), "an excluded row is a training row"
print(f"\nprotected rows excluded (same ids for CI and CII): {pe['n_excluded']}")
by = pe["by_split_class"]
print(pd.DataFrame({f"{sp} excl/rows": {k: f"{v['excluded']}/{v['rows']}" for k, v in c.items()}
                    for sp, c in by.items()}).to_string())
print("\ncause = first transform step that merged the rows (per scenario scaler):")
for sc, x in pe["diagnosis"].items():
    for sp, d in x.items():
        print(f"  {sc}/{sp}: {d['first_step']} | same label {d['same_label_as_train']}, "
              f"different {d['different_label_from_train']}")
        print(f"      top features that differed: {list(d['features_that_differed'].items())[:5]}")
bv, bt = by["val"]["BENIGN"]["kept"], by["test"]["BENIGN"]["kept"]
print(f"\nbenign kept for false alarms: val {bv:,} (step {1/bv:.5f}), test {bt:,}")

## 7. Shapes and the Stage 2 tokenizer self-test (no training)

In [ ]:
sys.path.insert(0, CODE_DIR)
from s1b_experiences import apply_scaler
class_map = json.loads((OUT / "class_map.json").read_text())
scaler = json.loads((OUT / "cii" / "scaler.json").read_text())
z = np.load(OUT / "cii" / "experience_rids.npz")
trf = pd.read_parquet(OUT / "clean_train.parquet")
cur, old = trf[trf["_rid"].isin(z["e1"])], trf[trf["_rid"].isin(z["e0"])]
batch = pd.concat([cur.sample(min(192, len(cur)), random_state=0), old.sample(min(64, len(old)), random_state=0)])
x_num, x_cat, y = apply_scaler(batch, scaler, class_map)
F, C, d = x_num.shape[1], x_cat.shape[1], 32
print("batch x_num:", x_num.shape, "x_cat:", x_cat.shape, f"| audited feature count F={F}")
print(f"model tokens: ({len(batch)}, 1 CLS + {F} + {C}, {d}) = ({len(batch)}, {1+F+C}, {d})")
try:
    import torch
    from s2_model import TabTransformer
    card = [len(v) + 1 for v in scaler["cat_vocab"].values()]
    m = TabTransformer(n_num=F, n_classes=2, cat_cardinalities=card, d=32, heads=4, layers=2).eval()  # eval: no dropout
    xn, xc = torch.tensor(x_num), torch.tensor(x_cat)
    print("token tensor:", tuple(m.tokenizer(xn, xc).shape))
    before = m(xn, xc).detach().clone()
    m.expand_head(3)
    assert torch.allclose(before, m(xn, xc).detach()[:, :2], atol=1e-6)
    print("head expansion preserved old logits: OK")
except ImportError:
    print("torch not installed here; on Kaggle this cell runs the check.")